In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [2]:
games_details = pd.read_csv("../data/processed/games_details_cleaned.csv")

C:\Users\JB MADHUBALA\AppData\Local\Temp\ipykernel_35220\2116805495.py:1: DtypeWarning: Columns (6,9) have mixed types. Specify dtype option on import or set low_memory=False.
  games_details = pd.read_csv("../data/processed/games_details_cleaned.csv")


In [3]:
games = pd.read_csv("../data/processed/games_with_teams.csv", parse_dates=["GAME_DATE_EST"])

games.head()

,GAME_DATE_EST,GAME_ID,GAME_STATUS_TEXT,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,TEAM_ID_home,PTS_home,FG_PCT_home,FT_PCT_home,FG3_PCT_home,AST_home,REB_home,TEAM_ID_away,PTS_away,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS,GAME_RESULT,TOTAL_POINTS,LEAGUE_ID_x,MIN_YEAR_x,MAX_YEAR_x,ABBREVIATION_x,HOME_TEAM_NAME,YEARFOUNDED_x,HOME_CITY,ARENA_x,ARENACAPACITY_x,OWNER_x,GENERALMANAGER_x,HEADCOACH_x,DLEAGUEAFFILIATION_x,LEAGUE_ID_y,MIN_YEAR_y,MAX_YEAR_y,ABBREVIATION_y,AWAY_TEAM_NAME,YEARFOUNDED_y,AWAY_CITY,ARENA_y,ARENACAPACITY_y,OWNER_y,GENERALMANAGER_y,HEADCOACH_y,DLEAGUEAFFILIATION_y
0,2022-12-22,22200477,Final,1610612740,1610612759,2022,1610612740,126.0,0.484,0.926,0.382,25.0,46.0,1610612759,117.0,0.478,0.815,0.321,23.0,44.0,1,HOME_WIN,243.0,0,2002,2019,NOP,Pelicans,2002,New Orleans,Smoothie King Center,NaN,Tom Benson,Trajan Langdon,Alvin Gentry,No Affiliate,0,1976,2019,SAS,Spurs,1976,San Antonio,AT&T Center,18694.0,Peter Holt,Brian Wright,Gregg Popovich,Austin Spurs
1,2022-12-22,22200478,Final,1610612762,1610612764,2022,1610612762,120.0,0.488,0.952,0.457,16.0,40.0,1610612764,112.0,0.561,0.765,0.333,20.0,37.0,1,HOME_WIN,232.0,0,1974,2019,UTA,Jazz,1974,Utah,Vivint Smart Home Arena,20148.0,Greg Miller,Dennis Lindsey,Quin Snyder,Salt Lake City Stars,0,1961,2019,WAS,Wizards,1961,Washington,Capital One Arena,20647.0,Ted Leonsis,Tommy Sheppard,Scott Brooks,Capital City Go-Go
2,2022-12-21,22200466,Final,1610612739,1610612749,2022,1610612739,114.0,0.482,0.786,0.313,22.0,37.0,1610612749,106.0,0.470,0.682,0.433,20.0,46.0,1,HOME_WIN,220.0,0,1970,2019,CLE,Cavaliers,1970,Cleveland,Quicken Loans Arena,20562.0,Dan Gilbert,Koby Altman,John Beilein,Canton Charge,0,1968,2019,MIL,Bucks,1968,Milwaukee,Fiserv Forum,17500.0,Wesley Edens & Marc Lasry,Jon Horst,Mike Budenholzer,Wisconsin Herd
3,2022-12-21,22200467,Final,1610612755,1610612765,2022,1610612755,113.0,0.441,0.909,0.297,27.0,49.0,1610612765,93.0,0.392,0.735,0.261,15.0,46.0,1,HOME_WIN,206.0,0,1949,2019,PHI,76ers,1949,Philadelphia,Wells Fargo Center,NaN,Joshua Harris,Elton Brand,Brett Brown,Delaware Blue Coats,0,1948,2019,DET,Pistons,1948,Detroit,Little Caesars Arena,21000.0,Tom Gores,Ed Stefanski,Dwane Casey,Grand Rapids Drive
4,2022-12-21,22200468,Final,1610612737,1610612741,2022,1610612737,108.0,0.429,1.000,0.378,22.0,47.0,1610612741,110.0,0.500,0.773,0.292,20.0,47.0,0,AWAY_WIN,218.0,0,1949,2019,ATL,Hawks,1949,Atlanta,State Farm Arena,18729.0,Tony Ressler,Travis Schlenk,Lloyd Pierce,Erie Bayhawks,0,1966,2019,CHI,Bulls,1966,Chicago,United Center,21711.0,Jerry Reinsdorf,Gar Forman,Jim Boylen,Windy City Bulls


In [4]:
# Select only required columns
games_clean = games[[
    "GAME_DATE_EST",
    "GAME_ID",
    "SEASON",
    "HOME_TEAM_NAME",
    "HOME_CITY",
    "AWAY_TEAM_NAME",
    "AWAY_CITY",
    "PTS_home",
    "PTS_away",
    "FG_PCT_home",
    "FG_PCT_away",
    "AST_home",
    "AST_away",
    "REB_home",
    "REB_away",
    "HOME_TEAM_WINS",
    "GAME_RESULT",
    "TOTAL_POINTS"
]].copy()

In [5]:
ranking = pd.read_csv("../data/processed/ranking_cleaned.csv")

ranking_clean = ranking[[
    "TEAM_ID",
    "SEASON_ID",
    "CONFERENCE"
]].copy()

In [6]:
games_clean["POINT_DIFF"] = games_clean["PTS_home"] - games_clean["PTS_away"]

In [7]:
games_clean["WINNING_TEAM"] = np.where(
    games_clean["HOME_TEAM_WINS"] == 1,
    games_clean["HOME_TEAM_NAME"],
    games_clean["AWAY_TEAM_NAME"]
)

In [8]:
games_clean["YEAR"] = games_clean["GAME_DATE_EST"].dt.year

In [9]:
games_clean["HOME_WIN"] = games_clean["HOME_TEAM_WINS"]
games_clean["AWAY_WIN"] = 1 - games_clean["HOME_TEAM_WINS"]

In [10]:
home_away_summary = games_clean.groupby("SEASON").agg(
    HOME_WIN_RATE=("HOME_WIN", "mean"),
    AVG_HOME_POINTS=("PTS_home", "mean"),
    AVG_AWAY_POINTS=("PTS_away", "mean")
).reset_index()

home_away_summary.head()

,SEASON,HOME_WIN_RATE,AVG_HOME_POINTS,AVG_AWAY_POINTS
0,2003,0.622084,94.907465,91.092535
1,2004,0.604993,98.604993,95.381791
2,2005,0.604749,98.406425,95.172486
3,2006,0.591261,99.849190,96.944327
4,2007,0.610914,101.273565,97.549256


In [11]:
team_stats = games_clean.groupby(["SEASON", "HOME_TEAM_NAME"]).agg(
    GAMES_PLAYED=("GAME_ID", "count"),
    AVG_POINTS_SCORED=("PTS_home", "mean"),
    AVG_POINTS_CONCEDED=("PTS_away", "mean"),
    WIN_RATE=("HOME_TEAM_WINS", "mean")
).reset_index()

team_stats.head()

,SEASON,HOME_TEAM_NAME,GAMES_PLAYED,AVG_POINTS_SCORED,AVG_POINTS_CONCEDED,WIN_RATE
0,2003,76ers,41,88.439024,87.560976,0.512195
1,2003,Bucks,43,100.534884,97.162791,0.627907
2,2003,Bulls,41,88.902439,93.682927,0.341463
3,2003,Cavaliers,41,95.536585,94.609756,0.560976
4,2003,Celtics,44,95.113636,96.181818,0.431818


In [12]:
season_summary = games_clean.groupby("SEASON").agg(
    AVG_POINTS=("TOTAL_POINTS", "mean"),
    HOME_WIN_RATE=("HOME_TEAM_WINS", "mean"),
    TOTAL_GAMES=("GAME_ID", "count")
).reset_index()

In [13]:
games_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 558746 entries, 0 to 558745
Data columns (total 30 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   GAME_ID         558746 non-null  int64  
 1   TEAM_ID         558746 non-null  int64  
 2   Team            558746 non-null  object 
 3   TEAM_CITY       558746 non-null  object 
 4   PLAYER_ID       558746 non-null  int64  
 5   Player          558746 non-null  object 
 6   NICKNAME        43392 non-null   object 
 7   START_POSITION  558746 non-null  object 
 8   COMMENT         558746 non-null  object 
 9   MIN             558746 non-null  object 
 10  FGM             558746 non-null  float64
 11  FGA             558746 non-null  float64
 12  FG_PCT          558746 non-null  float64
 13  FG3M            558746 non-null  float64
 14  FG3A            558746 non-null  float64
 15  FG3_PCT         558746 non-null  float64
 16  FTM             558746 non-null  float64
 17  FTA       

In [14]:
games_details = games_details.merge(
    games_clean[["GAME_ID", "SEASON"]],
    on="GAME_ID",
    how="left"
)

In [15]:
player_summary = games_details.groupby(
    ["SEASON", "Player"]
).agg(
    AVG_POINTS=("Points", "mean"),
    AVG_REB=("REB", "mean"),
    AVG_AST=("AST", "mean"),
    GAMES_PLAYED=("GAME_ID", "count")
).reset_index()

player_summary = player_summary[player_summary["GAMES_PLAYED"] >= 20]

In [16]:
games_clean.to_csv("../data/processed/games_features.csv", index=False)
team_stats.to_csv("../data/processed/team_stats.csv", index=False)
home_away_summary.to_csv("../data/processed/home_away_summary.csv", index=False)
season_summary.to_csv("../data/processed/season_summary.csv", index=False)
player_summary.to_csv("../data/processed/player_summary.csv", index=False)
